# TensorFlow/Keras 이미지 분류

TensorFlow 2.x와 Keras를 사용한 이미지 분류 모델 구축

## 학습 목표
1. Keras Sequential/Functional API 사용법
2. CNN 모델 구축
3. Transfer Learning
4. 데이터 증강

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 1. 데이터 로드 (CIFAR-10)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# 정규화
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 클래스 이름
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training data: {x_train.shape}")
print(f"Test data: {x_test.shape}")

In [ ]:
# 샘플 이미지 시각화
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Sequential API로 모델 구축

In [ ]:
model_sequential = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_sequential.summary()

## 3. Functional API로 모델 구축

In [ ]:
def create_model_functional():
    inputs = layers.Input(shape=(32, 32, 3))
    
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    
    return models.Model(inputs, outputs)

model = create_model_functional()
model.summary()

## 4. 모델 컴파일 및 학습

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 콜백 설정
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
]

history = model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=callbacks
)

In [ ]:
# 학습 결과 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'], label='Train')
ax1.plot(history.history['val_loss'], label='Validation')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train')
ax2.plot(history.history['val_accuracy'], label='Validation')
ax2.set_title('Accuracy')
ax2.legend()

plt.show()

# 테스트 평가
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"\nTest accuracy: {test_acc:.4f}")

## 5. Transfer Learning (ResNet50)

In [ ]:
# 사전 학습된 모델 로드
base_model = keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# 기본 모델 동결
base_model.trainable = False

# 새로운 분류기 추가
inputs = layers.Input(shape=(224, 224, 3))
x = keras.applications.resnet50.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

transfer_model = models.Model(inputs, outputs)
transfer_model.summary()

## 6. 데이터 증강

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# 증강 예시 시각화
plt.figure(figsize=(10, 5))
for i in range(8):
    augmented = data_augmentation(x_train[:1])
    plt.subplot(2, 4, i + 1)
    plt.imshow(augmented[0])
    plt.axis('off')
plt.suptitle('Data Augmentation Examples')
plt.show()

## 연습 문제

1. VGG16, MobileNet 등 다른 사전 학습 모델을 사용해보세요.
2. Fine-tuning: 기본 모델의 일부 레이어를 동결 해제하고 학습해보세요.
3. 다른 데이터셋 (Fashion-MNIST, Dogs vs Cats)으로 학습해보세요.